In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kag\gle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

<>:12: SyntaxWarning: invalid escape sequence '\g'
<>:12: SyntaxWarning: invalid escape sequence '\g'
/tmp/ipykernel_930/2204704819.py:12: SyntaxWarning: invalid escape sequence '\g'
  for dirname, _, filenames in os.walk('/kag\gle/input'):


In [15]:
import pandas as pd
import sqlite3
import os

file_path = '/kaggle/input/datasets/binib1997/superstore/Superstore.csv'

In [16]:
df = pd.read_csv(file_path, encoding='windows-1252')
df.info()
# df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [17]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [18]:
print(df.isnull().sum())

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [19]:
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Month'] = df['Order_Date'].dt.month
df['Processing_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days

df['Postal_Code'] = df['Postal_Code'].astype(str)
df['Row_ID'] = df['Row_ID'].astype(str)
numeric_measures = ['Sales', 'Quantity', 'Discount', 'Profit']

In [21]:
region_targets = pd.DataFrame({
    'Region': ['East', 'West', 'Central', 'South'],
    'Sales_Target': [700000, 750000, 500000, 450000]
})

df_merged = df.merge(region_targets, on='Region', how='left')
df_merged[['Region', 'Sales', 'Sales_Target']].head()

,Region,Sales,Sales_Target
0,South,261.9600,450000
1,South,731.9400,450000
2,West,14.6200,750000
3,South,957.5775,450000
4,South,22.3680,450000


##### df_merged['Region'].value_counts()

In [22]:
# First Question (What is the top 5 orders by sales) by pandas 
top_5_pandas = df[['Order_ID', 'Customer_Name', 'Sales']].sort_values(by='Sales', ascending=False).head(5)
print("Top 5 :\n", top_5_pandas)

print('\n')
# Second Question (How many orders that .belongs to Furniture category) by pandas
furniture_count_pandas = df[df['Category'] == 'Furniture'].shape[0]
print(f"Orders in Furniture: {furniture_count_pandas} orders")

print('\n')
# Third Question (Total Sales by Region) by pandas
sales_by_region_pandas = df.groupby('Region')['Sales'].sum().reset_index()
sales_by_region_pandas = sales_by_region_pandas.sort_values(by='Sales', ascending=False)
print("Sales by Region :\n", sales_by_region_pandas)

print('\n')
# Fourth Question (Average profit by sub-catecory) by pandas
avg_profit_pandas = df.groupby('Sub_Category')['Profit'].mean().reset_index()
avg_profit_pandas = avg_profit_pandas.sort_values(by='Profit', ascending=False)
print("Avg profit by sub-category:\n " ,avg_profit_pandas.head())

print('\n')
# Fifth Question (Percentage of losing orders (Profit < 0)) by pandas
total_orders = len(df)
losing_orders_count = (df['Profit'] < 0).sum()
loss_percentage_pandas = (losing_orders_count / total_orders) * 100
print(f"Percent of losing Orders : {loss_percentage_pandas:.2f}%")

print('\n')
# Sixth Question (Relation between discount and average profit) by pandas
discount_profit_pandas = df.groupby('Discount')['Profit'].mean().reset_index()
discount_profit_pandas = discount_profit_pandas.sort_values(by='Discount')
print("Relation Result:")
print(discount_profit_pandas)

print('\n')
# Seventh Question (Average Shipping Days by Ship Mode)
avg_shipping_pandas = df.groupby('Ship_Mode')['Processing_Days'].mean().reset_index()
avg_shipping_pandas = avg_shipping_pandas.sort_values(by='Processing_Days')
print("Average Shipping Days:")
print(avg_shipping_pandas)

print('\n')
# Eighth Question (Most frequent customer segment (by order count)) by pandas
top_segment_pandas = df['Segment'].value_counts().reset_index()
top_segment_pandas.columns = ['Segment', 'Order_Count']
print("Most Frequent Customer Segment:")
print(top_segment_pandas)

print('\n')
# Ninth Question (Sum of the orders by year) by pandas
sales_by_year = df.groupby('Order_Year')['Sales'].sum().reset_index()
sales_by_year = sales_by_year.sort_values(by='Sales', ascending=False)
print("Sales By Year:")
print(sales_by_year)

print('\n')
# Tenth Question( The month with the highest number of orders) by pandas
orders_by_month = df.groupby('Order_Month').size().reset_index(name='Order_Count')
orders_by_month = orders_by_month.sort_values(by='Order_Count', ascending=False)
print("Orders By Month: ")
print(orders_by_month)


Top 5 :
             Order_ID Customer_Name      Sales
2697  CA-2014-145317   Sean Miller  22638.480
6826  CA-2016-118689  Tamara Chand  17499.950
8153  CA-2017-140151  Raymond Buch  13999.960
2623  CA-2017-127180  Tom Ashbrook  11199.968
4190  CA-2017-166709  Hunter Lopez  10499.970


Orders in Furniture: 2121 orders


Sales by Region :
     Region        Sales
3     West  725457.8245
1     East  678781.2400
0  Central  501239.8908
2    South  391721.9050


Avg profit by sub-category:
     Sub_Category      Profit
6       Copiers  817.909190
0   Accessories   54.111788
13       Phones   50.073938
5        Chairs   43.095894
1    Appliances   38.922758


Percent of losing Orders : 18.72%


Relation Result:
    Discount      Profit
0       0.00   66.900292
1       0.10   96.055074
2       0.15   27.288298
3       0.20   24.702572
4       0.30  -45.679636
5       0.32  -88.560656
6       0.40 -111.927429
7       0.45 -226.646464
8       0.50 -310.703456
9       0.60  -43.077212
10      0

In [23]:
db_path = '/kaggle/working/superstore.db'
conn = sqlite3.connect(db_path)
df.to_sql('orders', conn, if_exists='replace', index=False)

9994

In [24]:
test = pd.read_sql("SELECT * FROM orders LIMIT 5", conn)
print(test)

  Row_ID        Order_ID           Order_Date            Ship_Date  \
0      1  CA-2016-152156  2016-11-08 00:00:00  2016-11-11 00:00:00   
1      2  CA-2016-152156  2016-11-08 00:00:00  2016-11-11 00:00:00   
2      3  CA-2016-138688  2016-06-12 00:00:00  2016-06-16 00:00:00   
3      4  US-2015-108966  2015-10-11 00:00:00  2015-10-18 00:00:00   
4      5  US-2015-108966  2015-10-11 00:00:00  2015-10-18 00:00:00   

        Ship_Mode Customer_ID    Customer_Name    Segment        Country  \
0    Second Class    CG-12520      Claire Gute   Consumer  United States   
1    Second Class    CG-12520      Claire Gute   Consumer  United States   
2    Second Class    DV-13045  Darrin Van Huff  Corporate  United States   
3  Standard Class    SO-20335   Sean O'Donnell   Consumer  United States   
4  Standard Class    SO-20335   Sean O'Donnell   Consumer  United States   

              City  ...         Category Sub_Category  \
0        Henderson  ...        Furniture    Bookcases   
1       

In [25]:
count = pd.read_sql("SELECT COUNT(*) as total FROM orders", conn)
print(count)

   total
0   9994


In [27]:
# First Question (What is the top 5 orders by sales) by SQL 
query_1 = """
    SELECT Order_ID, Customer_Name, Sales 
    FROM orders 
    ORDER BY Sales DESC 
    LIMIT 5
"""
top_5_sql = pd.read_sql(query_1, conn)
print("Top 5 :\n", top_5_sql)

print('\n')
# Second Question (How many orders that .belongs to Furniture category) by SQL
query_2 = "SELECT COUNT(*) as Count FROM orders WHERE Category = 'Furniture'"
furniture_count_sql = pd.read_sql(query_2, conn)
print(f"Orders in Furniture: {furniture_count_sql['Count'].iloc[0]} orders")

print('\n')
# Third Question (Total Sales by Region) by SQL
query_3 = """
    SELECT Region, SUM(Sales) as Total_Sales 
    FROM orders 
    GROUP BY Region
    ORDER BY Total_Sales DESC
"""
sales_by_region_sql = pd.read_sql(query_3, conn)
print("Sales by Region :\n", sales_by_region_sql)

print('\n')
# Fourth Question (Average profit by sub-catecory) by SQL
query_4 = """
    SELECT Sub_Category, AVG(Profit) as Avg_Profit 
    FROM orders 
    GROUP BY Sub_Category
    ORDER BY Avg_Profit DESC
"""
avg_profit_sql = pd.read_sql(query_4, conn)
print("Avg profit by sub-category:\n " ,avg_profit_sql.head())

print('\n')
# Fifth Question (Percentage of losing orders (Profit < 0)) by SQL
query_5 = """
    SELECT 
        (SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) as Loss_Percentage
    FROM orders
"""
loss_percentage_sql = pd.read_sql(query_5, conn)
print(f"Percent of losing Orders: {loss_percentage_sql['Loss_Percentage'].iloc[0]:.2f}%")

print('\n')
# Sixth Question (Relation between discount and average profit) by SQL
query_6 = """
    SELECT Discount, AVG(Profit) as Avg_Profit 
    FROM orders 
    GROUP BY Discount
    ORDER BY Discount ASC
"""
discount_profit_sql = pd.read_sql(query_6, conn)
print("Relation Result:")
print(discount_profit_sql)

print('\n')
# Seventh Question (Average Shipping Days by Ship Mode) by SQL
query_7 = """
    SELECT 
        Ship_Mode, 
        AVG(julianday(Ship_Date) - julianday(Order_Date)) as Avg_Shipping_Days 
    FROM orders 
    GROUP BY Ship_Mode
    ORDER BY Avg_Shipping_Days ASC
"""
avg_shipping_sql = pd.read_sql(query_7, conn)
print("Average Shipping Days:")
print(avg_shipping_sql)

print('\n')
# Eighth Question (Most frequent customer segment (by order count)) by SQL
query_8 = """
    SELECT Segment, COUNT(*) as Order_Count 
    FROM orders 
    GROUP BY Segment
    ORDER BY Order_Count DESC
"""
top_segment_sql = pd.read_sql(query_8, conn)
print("Most Frequent Customer Segment:")
print(top_segment_sql)

print('\n')
# Ninth Question (Sum of the orders by year) by SQL
query_9 = """
    SELECT Order_Year, SUM(Sales) as Total_Sales
    From orders
    GROUP BY Order_Year
    ORDER BY Order_Year
"""
sales_by_year = pd.read_sql(query_9, conn)
print("Sales By Year:")
print(sales_by_year)

print('\n')
# Tenth Question( The month with the highest number of orders) by SQL
query_10 = """
    SELECT Order_Month, COUNT(*) as Order_Count 
    From orders
    GROUP BY Order_Month
    ORDER BY Order_Count DESC
"""
orders_by_month = pd.read_sql(query_10, conn)
print("Orders By Month: ")
print(orders_by_month)

Top 5 :
          Order_ID Customer_Name      Sales
0  CA-2014-145317   Sean Miller  22638.480
1  CA-2016-118689  Tamara Chand  17499.950
2  CA-2017-140151  Raymond Buch  13999.960
3  CA-2017-127180  Tom Ashbrook  11199.968
4  CA-2017-166709  Hunter Lopez  10499.970


Orders in Furniture: 2121 orders


Sales by Region :
     Region  Total_Sales
0     West  725457.8245
1     East  678781.2400
2  Central  501239.8908
3    South  391721.9050


Avg profit by sub-category:
    Sub_Category  Avg_Profit
0      Copiers  817.909190
1  Accessories   54.111788
2       Phones   50.073938
3       Chairs   43.095894
4   Appliances   38.922758


Percent of losing Orders: 18.72%


Relation Result:
    Discount  Avg_Profit
0       0.00   66.900292
1       0.10   96.055074
2       0.15   27.288298
3       0.20   24.702572
4       0.30  -45.679636
5       0.32  -88.560656
6       0.40 -111.927429
7       0.45 -226.646464
8       0.50 -310.703456
9       0.60  -43.077212
10      0.70  -95.874060
11      0